In [0]:
# ==========================================
# PIPELINE 01: AMERICAN CHEESE PRICE PROCESSING
# ==========================================

In [0]:
# Import the Pandas library for data processing and analysis
import pandas as pd

# Define the file path for the raw CSV stored in the Databricks Volume
file_path = "/Volumes/workspace/default/american_cheese_price_data/costinflation-american-cheese-retail-prices-raw-2026-07-13-to-2026-08-10.csv"

# Load the raw CSV file into a Pandas DataFrame
df = pd.read_csv(file_path) 

# Check the number of rows and columns in the dataset
print(df.shape)

# Review the data types of all columns
print(df.dtypes)

# Display the first five records for an initial data preview
df.head(5)

(12039, 16)
series_id                     object
series_title                  object
canonical_url                 object
geography_type                object
geography_id                   int64
geography_label               object
observed_date                 object
product_name                  object
quantity_value               float64
quantity_unit                 object
quantity_name                 object
price_amount                 float64
currency_code                 object
normalized_price_amount      float64
normalized_quantity_value    float64
normalized_quantity_unit      object
dtype: object


,series_id,series_title,canonical_url,geography_type,geography_id,geography_label,observed_date,product_name,quantity_value,quantity_unit,quantity_name,price_amount,currency_code,normalized_price_amount,normalized_quantity_value,normalized_quantity_unit
0,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,"Amazon Grocery, Pasteurized Process American C...",16.0,ounces,16 ounces,2.48,USD,0.9300,0.375,pounds
1,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,"Amazon Saver, American Singles, Pasteurized Pr...",16.0,ounces,16 ounces,2.48,USD,0.9300,0.375,pounds
2,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,"American cheese slices , contains 9 ingredient...",32.0,ounces,32 ounces,42.99,USD,8.0606,0.375,pounds
3,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,BORDEN CHEESE SLICES AMERICAN SINGLES 12 OZ PA...,36.0,ounces,36 ounces,49.90,USD,8.3167,0.375,pounds
4,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,Borden Lactose Free American Singles 8 Oz (Pac...,24.0,ounces,24 ounces,49.90,USD,12.4750,0.375,pounds


Data Quality Check

In [0]:
# NULL's Per columns 
print(df.isnull().sum())

# Count potential duplicate records based on the selected business key columns
dupe_count = df.duplicated(subset=["geography_id", "observed_date", "product_name", "price_amount"]).sum()
print(f"Extracted Duplicate Rows:{dupe_count}")

#Parse Data properly (string to datetime)
df["observed_date"] = pd.to_datetime(df["observed_date"])
print(df["observed_date"].min(), "to" , df["observed_date"].max())

# Price Check
print(df["price_amount"].describe()) 
outliers = df[(df["price_amount"]<= 0) | (df["price_amount"]>100)]
print(f"Outliers amount: {len(outliers)}")

# check NULL in normalized priced amount
print(df[df["normalized_price_amount"].isnull()]["quantity_unit"].value_counts())

series_id                       0
series_title                    0
canonical_url                   0
geography_type                  0
geography_id                    0
geography_label                 0
observed_date                   0
product_name                    0
quantity_value                  0
quantity_unit                   0
quantity_name                   0
price_amount                    0
currency_code                   0
normalized_price_amount      1033
normalized_quantity_value       0
normalized_quantity_unit        0
dtype: int64
Extracted Duplicate Rows:0
2026-07-13 00:00:00 to 2026-08-10 00:00:00
count    12039.000000
mean        40.729366
std         21.701374
min          1.180000
25%         29.990000
50%         42.990000
75%         49.990000
max         89.990000
Name: price_amount, dtype: float64
Outliers amount: 0
quantity_unit
unknown    1033
Name: count, dtype: int64


Handle Nulls in dataframe

In [0]:
null_rows = df[df["normalized_price_amount"].isnull()]
print(df["quantity_unit"].value_counts())
print(df["quantity_value"].value_counts()) 

# keep the rows but flag which one has usabl prepound value
df["has_normalized_value"] = df["normalized_price_amount"].notnull()
print(df["has_normalized_value"].value_counts())

quantity_unit
ounces     6502
pounds     4504
unknown    1033
Name: count, dtype: int64
quantity_value
32.00     2443
44.00     1382
5.00      1356
6.00      1119
0.00      1033
3.00       998
16.00      682
2.00       680
4.12       348
64.00      347
36.00      347
80.00      346
4.00       343
24.00      340
8.00       198
12.00       65
432.00      12
Name: count, dtype: int64
has_normalized_value
True     11006
False     1033
Name: count, dtype: int64


Standard product name and brand form

In [0]:
# Define the list of known brands to identify from product names
brands = ["Kraft", "Borden", "Amazon", "James Farm", "Deli Deluxe", "Great Value", "Land O'Lakes"]

# Extract the first matching brand name from the product name
def extract_brand(name):
    for b in brands:
        if b.lower() in name.lower():
            return b
    return "Unknown Brand"

# Apply the brand extraction logic to create a new brand column
df["brand"] = df["product_name"].apply(extract_brand)

# Review the distribution of extracted brands
print(df["brand"].value_counts())


# Categorize each product based on keywords found in the product name
def extract_form(name):
    name_lower =name.lower()
    if "slice" in name_lower:
        return "sliced"
    elif "shred" in name_lower:
        return "shredded"
    elif "block" in name_lower or "loaf" in name_lower:
        return "block"
    else:
        return "unspecified"

# Apply the cheese form extraction function to create a new derived column    
df["cheese_form"] =df["product_name"].apply(extract_form)

# Review the distribution of cheese forms
print(df["cheese_form"].value_counts())

brand
Unknown Brand    7584
Deli Deluxe      1386
Kraft            1138
Borden            716
James Farm        693
Amazon            522
Name: count, dtype: int64
cheese_form
sliced         11325
unspecified      714
Name: count, dtype: int64


Aggregation

In [0]:
# Derive the starting date of each week from the observation date
df["Week"] = df["observed_date"].dt.to_period("W").dt.start_time

# Calculate the average normalized price by city and week
city_week_avg = (
    df[df["has_normalized_value"]]
    .groupby(["geography_label","Week"])["normalized_price_amount"]
    .mean()
    .reset_index()
    .rename(columns={"normalized_price_amount":"avg_price_per_lb"}))

# Preview the aggregated weekly city-level results
print(city_week_avg.head(10)) 

# Calculate price statistics for each extracted brand
brand_avg =(
    df[df["has_normalized_value"]]
    .groupby("brand")["normalized_price_amount"]
    .agg(["mean","min","max","count"])
    .reset_index()
    .sort_values("mean",ascending=False)
)

# Display the brand-level price statistics
print(brand_avg)

                       geography_label       Week  avg_price_per_lb
0  Chicago, IL - North Center/Avondale 2026-07-13          5.704721
1  Chicago, IL - North Center/Avondale 2026-07-20          5.766114
2  Chicago, IL - North Center/Avondale 2026-07-27          5.727330
3  Chicago, IL - North Center/Avondale 2026-08-03          5.689979
4  Chicago, IL - North Center/Avondale 2026-08-10          5.717471
5                          Clayton, MO 2026-07-13          6.453627
6                          Clayton, MO 2026-07-20          6.342126
7                          Clayton, MO 2026-07-27          6.226023
8                          Clayton, MO 2026-08-03          6.145386
9                          Clayton, MO 2026-08-10          6.152703
           brand       mean     min      max  count
1         Borden  10.049744  1.1950  12.4750    716
2    Deli Deluxe   7.451902  5.6231   9.1856   1386
5  Unknown Brand   5.745706  0.0347  10.1944   7131
4          Kraft   5.728953  1.4950   7.6875

Save as Delta Table

In [0]:
# Convert the cleaned and enriched Pandas DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(df)

# Save the cleaned and enriched dataset as a managed Delta table
spark_df.write.format("delta").mode("overwrite").saveAsTable("american_cheese_cleaned_delta_table")

# Display the contents of the saved Delta table
display(spark.sql("select * from american_cheese_cleaned_delta_table limit 10"))

# Validate the total number of records written to the Delta table
print(spark.sql("select count(*) as row_count from american_cheese_cleaned_delta_table").collect())

series_id,series_title,canonical_url,geography_type,geography_id,geography_label,observed_date,product_name,quantity_value,quantity_unit,quantity_name,price_amount,currency_code,normalized_price_amount,normalized_quantity_value,normalized_quantity_unit,brand,cheese_form,Week,has_normalized_value
american_cheese,American cheese,https://costinflation.com/indices/american-cheese-price-history,postal_code,39206,"Jackson, MS - North Jackson",2026-08-03T00:00:00.000Z,"Yellow American Cheese, Sliced Cheese , Yellow Deli American is the staple for lunchtime sandwiches, gooey grilled cheese and juicy burgers. It's the ultimate sandwich secret weapon. Perfect for a classic ham and cheese sandwich , Sliced Deli American delivers gooey, melty, cheesy perfection to your favorite sandwiches, burgers and wraps., 72 slices [ 44 oz , 2.75 lb ]",44.0,ounces,44 ounces,43.95,USD,5.9932,0.375,pounds,Unknown Brand,sliced,2026-08-03T00:00:00.000Z,true
american_cheese,American cheese,https://costinflation.com/indices/american-cheese-price-history,postal_code,60618,"Chicago, IL - North Center/Avondale",2026-08-03T00:00:00.000Z,"Amazon Grocery, Pasteurized Process American Cheese, 16 Oz, 24 Ct",16.0,ounces,16 ounces,2.48,USD,0.93,0.375,pounds,Amazon,unspecified,2026-08-03T00:00:00.000Z,true
american_cheese,American cheese,https://costinflation.com/indices/american-cheese-price-history,postal_code,60618,"Chicago, IL - North Center/Avondale",2026-08-03T00:00:00.000Z,"Amazon Saver, American Singles, Pasteurized Prepared Cheese Product, 24 Slices, 16 Oz",16.0,ounces,16 ounces,2.48,USD,0.93,0.375,pounds,Amazon,sliced,2026-08-03T00:00:00.000Z,true
american_cheese,American cheese,https://costinflation.com/indices/american-cheese-price-history,postal_code,60618,"Chicago, IL - North Center/Avondale",2026-08-03T00:00:00.000Z,"American cheese slices , contains 9 ingredients or more, you can get the melt you love and the taste you crave with only 100% natural cheese. It’s a new standard that meets your expectations and makes a difference you can taste. This package contains 48 slices and is perfect for grilled cheese, burgers, and breakfast sandwiches. [ 32 oz , 2 lb ]",32.0,ounces,32 ounces,42.99,USD,8.0606,0.375,pounds,Unknown Brand,sliced,2026-08-03T00:00:00.000Z,true
american_cheese,American cheese,https://costinflation.com/indices/american-cheese-price-history,postal_code,60618,"Chicago, IL - North Center/Avondale",2026-08-03T00:00:00.000Z,BORDEN CHEESE SLICES AMERICAN SINGLES 12 OZ PACK OF 3,36.0,ounces,36 ounces,49.9,USD,8.3167,0.375,pounds,Borden,sliced,2026-08-03T00:00:00.000Z,true
american_cheese,American cheese,https://costinflation.com/indices/american-cheese-price-history,postal_code,60618,"Chicago, IL - North Center/Avondale",2026-08-03T00:00:00.000Z,Borden Lactose Free American Singles 8 Oz (Pack of 3),24.0,ounces,24 ounces,49.9,USD,12.475,0.375,pounds,Borden,unspecified,2026-08-03T00:00:00.000Z,true
american_cheese,American cheese,https://costinflation.com/indices/american-cheese-price-history,postal_code,60618,"Chicago, IL - North Center/Avondale",2026-08-03T00:00:00.000Z,"Deli Deluxe American Cheese Slices , elevate your favorite foods with big cheese energy. Try our slices on sandwiches, burgers, grilled cheeses and more. Sliced American cheese has a mild, slightly tangy flavor and smooth texture that goes well with meat and vegetable dishes, hot and cold. Pre-sliced for your convenience, our deli-style American cheese is perfect for layering on a classic sandwich, building a better burger or snacking straight from the package. [ 32 oz , 2 lb ] 48 ct.",32.0,ounces,32 ounces,29.99,USD,5.6231,0.375,pounds,Deli Deluxe,sliced,2026-08-03T00:00:00.000Z,true
american_cheese,American cheese,https://costinflation.com/indices/american-cheese-price-history,postal_code,60618,"Chicago, IL - North Center/Avondale",2026-08-03T00:00:00.000Z,"Deli Deluxe American Cheese Slices , elevate your favorite foods with big cheese energy. Try our slices on sandwiches, bu

[Row(row_count=12039)]


In [0]:
# Convert the city-week Pandas aggregation to Spark and save it as a Delta table
spark.createDataFrame(city_week_avg).write.format("delta").mode("overwrite").saveAsTable("city_week_avg_delta_table")

# Convert the brand-level Pandas aggregation to Spark and save it as a Delta table
spark.createDataFrame(brand_avg).write.format("delta").mode("overwrite").saveAsTable("brand_avg_delta_table")